<a href="https://colab.research.google.com/github/yanapatt/Mushrooms-Classification/blob/main/%5BCNN%5D_ResNet50_MCG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CNN Benchmark**

## **Section 0: Import important library**

In [ ]:
%%capture
!pip install torchmetrics
!pip install torchsummary
!pip install pkbar
!pip install calflops

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchmetrics.functional import accuracy, f1_score, precision, recall
import torchvision
from torchsummary import summary
import calflops

import numpy as np
import random as rd
import pandas as pd

import sklearn
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

import os
from collections import defaultdict
import itertools
import pathlib
import json
import zipfile
import shutil

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
import PIL
import pkbar

from time import time

In [ ]:
def set_seed(seed_num=42):
    rd.seed(seed_num)
    np.random.seed(seed_num)
    torch.manual_seed(seed_num)
    os.environ['PYTHONHASHSEED']=str(seed_num)
    torch.cuda.manual_seed(seed_num)
    torch.cuda.manual_seed_all(seed_num)

    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark=False
        torch.backends.cudnn.deterministic=True
    print(f"Global seed has been set at {seed_num}")

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    rd.seed(worker_seed)

def get_generator(seed_num=42):
    g=torch.Generator()
    g.manual_seed(seed_num)
    return g

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"You are running on {device}.")

SEED = 2025
set_seed(SEED)
g = get_generator(SEED)

## **Section 1: Setting directory and preparing the dataset**

**Drive Directory**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
drive_dir = "/content/drive/MyDrive/projectMushroom" # ตำแหน่ง Root Project Folder ใน G-Drive
dataset_drive_dir = f"{drive_dir}/data/mushroomCommonGenus"
dataset_name = "MCG"

model_name = "resnet50"
model_dir = os.path.join(drive_dir, "modelWeights", f"{model_name}{dataset_name}")
model_assets_dir = os.path.join(model_dir, "assets")

os.makedirs(model_dir, exist_ok=True)
os.makedirs(model_assets_dir, exist_ok=True)

font_dir = f"{drive_dir}/assets/fonts"
os.chdir(drive_dir)
print(f"Current directory: {os.getcwd()}")

In [ ]:
def set_matplotlib_font(font_dir, prefer_name=None):
    font_dir = pathlib.Path(font_dir)

    if not font_dir.exists():
        raise FileNotFoundError(f"Font directory not found: {font_dir}")

    font_files = font_manager.findSystemFonts(fontpaths=[str(font_dir)])

    if not font_files:
        raise RuntimeError(f"No font files found in {font_dir}")

    for font in font_files:
        font_manager.fontManager.addfont(font)

    selected_font = None
    for font in font_files:
        name = font_manager.FontProperties(fname=font).get_name()

        if prefer_name and prefer_name.lower() in name.lower():
            selected_font = name
            break

        if selected_font is None:
            selected_font = name

    matplotlib.rcParams['font.family'] = selected_font
    return selected_font

set_matplotlib_font(font_dir, prefer_name="Times-Roman")

**Runtime Directory**

In [ ]:
runtime_dir = "/content"
dataset_dir = os.path.join(runtime_dir, "mcgDataset")

os.makedirs(dataset_dir, exist_ok=True)

## **Section 2: Preparing the dataset**

**Load from G-Drive**

In [ ]:
with zipfile.ZipFile(f"{dataset_drive_dir}/mcgDataset.zip", "r") as zip_ref:
    zip_ref.extractall(dataset_dir)
print(f"Dataset has been extracted to {dataset_dir}")

In [ ]:
with open(f"{dataset_drive_dir}/class_to_idx.json", "r") as f:
    class_to_idx = json.load(f)

print(f"Class to index: {class_to_idx}")

with open(f"{dataset_drive_dir}/idx_to_class.json", "r") as f:
    idx_to_class = json.load(f)

print(f"Index to class: {idx_to_class}")

In [ ]:
BATCH_SIZE = 32
N_WORKERS = 2
IMG_SIZE = (224, 224)
INPUT_SHAPE = (3,) + IMG_SIZE

MEAN_NORM = [0.485,0.456,0.406]
STD_NORM = [0.229,0.224,0.225]

In [ ]:
train_transform = torchvision.transforms.Compose([
    torchvision.transforms.RandomResizedCrop(IMG_SIZE[0], scale=(0.8, 1.0)),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.RandomRotation(15),
    torchvision.transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=MEAN_NORM, std=STD_NORM)
])

transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),
    torchvision.transforms.CenterCrop(IMG_SIZE[0]),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=MEAN_NORM, std=STD_NORM)
])

In [ ]:
train_dataset = torchvision.datasets.ImageFolder(f"{dataset_dir}/train", transform=train_transform)
valid_dataset = torchvision.datasets.ImageFolder(f"{dataset_dir}/valid", transform=transform)
test_dataset = torchvision.datasets.ImageFolder(f"{dataset_dir}/test", transform=transform)

train_dataset.class_to_idx = class_to_idx
valid_dataset.class_to_idx = class_to_idx
test_dataset.class_to_idx = class_to_idx

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=N_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True
)

valid_loader = torch.utils.data.DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=N_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=N_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True
)

print("Train classes:", train_dataset.classes)
print("Train class_to_idx:", train_dataset.class_to_idx)

In [ ]:
CLASSES_NAME = train_dataset.classes
print(f"Classes name: {CLASSES_NAME}")
NUM_CLASSES = len(CLASSES_NAME)
print(f"Number of classes: {NUM_CLASSES}")

## **Section 2: Define CNNs architecture**

In [ ]:
model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model.fc.requires_grad = True

model = model.to(device)

In [ ]:
summary(model, INPUT_SHAPE, device=device.type)

## **Section 3: Define training loop, helper function, and evaluation function**

In [ ]:
MAX_EPOCHS = 15
BASE_LR = 0.001
TASK = "multiclass"
AVG = "macro"

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=BASE_LR, betas=(0.9, 0.999), eps=1e-08)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=3)

### **Define callback function**

In [ ]:
def save_model_state(save_path,
                     model,
                     epoch,
                     optimizer,
                     **kwargs
                     ):
    dir_path = os.path.dirname(save_path)
    if dir_path:
        os.makedirs(dir_path, exist_ok=True)

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }

    checkpoint.update(kwargs)
    torch.save(checkpoint, save_path)

def load_model_state(save_path,
                     model,
                     optimizer=None,
                     device="cpu"
                     ):
    if not os.path.exists(save_path):
        raise FileNotFoundError(f"Checkpoint not found: {save_path}")

    checkpoint = torch.load(save_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])

    if optimizer:
        if 'optimizer_state_dict' in checkpoint:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            for state in optimizer.state.values():
                for k, v in state.items():
                    if isinstance(v, torch.Tensor):
                        state[k] = v.to(device)
        else:
            print(f"[Warning]: Optimizer state not found in checkpoint at {save_path}")

    return checkpoint

In [ ]:
class ModelCheckpoint:
    def __init__(self,
                 optimizer,
                 save_path=None,
                 monitor="val_loss",
                 mode="min",
                 min_delta=0.0,
                 verbose=True
                 ):

        self.save_path = save_path
        self.optimizer = optimizer
        self.monitor = monitor
        self.mode = mode
        self.min_delta = min_delta
        self.verbose = verbose

        self.best_epoch = 0

        valid_modes = ["min", "max"]
        if mode not in valid_modes:
            raise ValueError(f"Mode must be one of {valid_modes}")

        if mode == "min":
            self.best_score = float("inf")
        else:
            self.best_score = float("-inf")

    def _is_improved(self, current_score):
        if self.mode == "min":
            return current_score < self.best_score - self.min_delta
        else:
            return current_score > self.best_score + self.min_delta

    def on_epoch_end(self, model, epoch, logs):
        current_score = logs.get(self.monitor)
        if current_score is None:
            raise ValueError(f"Monitor {self.monitor} not found in logs.")

        if self._is_improved(current_score):
            if self.verbose:
                print(f"Epoch: {epoch + 1} {self.monitor} improved from {self.best_score:.4f} to {current_score:.4f}, saving model to {self.save_path}")

            if self.save_path:
                save_model_state(
                    save_path=self.save_path,
                    model=model,
                    epoch=epoch + 1,
                    optimizer=self.optimizer,
                    monitor=self.monitor,
                    best_score=current_score
                )

            self.best_score = current_score
            self.best_epoch = epoch + 1
        else:
            if self.verbose:
                print(f"Epoch: {epoch + 1} {self.monitor} did not improve from {self.best_score:.4f}")

In [ ]:
class EarlyStopping:
    def __init__(self,
                 patience=5,
                 monitor="val_loss",
                 mode="min",
                 min_delta=0.0,
                 verbose=True
                 ):
        self.patience = patience
        self.monitor = monitor
        self.mode = mode
        self.min_delta = min_delta
        self.verbose = verbose

        self.counter = 0
        self.stop_training = False

        valid_modes = ["min", "max"]
        if mode not in valid_modes:
            raise ValueError(f"Mode must be one of {valid_modes}")

        if mode == "min":
            self.best_score = float("inf")
        else:
            self.best_score = -float("inf")

    def _is_improved(self, current_score):
        if self.mode == "min":
            return current_score < self.best_score - self.min_delta
        else:
            return current_score > self.best_score + self.min_delta

    def on_epoch_end(self, model, epoch, logs):
        current_score = logs.get(self.monitor)

        if current_score is None:
            raise ValueError(f"Monitor {self.monitor} not found in logs.")

        if self._is_improved(current_score):
            self.best_score = current_score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop_training = True
                if self.verbose:
                    print(f"Early stopping training at epoch {epoch + 1}")

### **Define perf_mertrics, training, eval, fit, and pred function**

In [ ]:
def performance_metrics(y_pred, y_true, num_classes, task="multiclass", avg="macro"):
    valid_tasks = ["binary", "multiclass"]
    if task not in valid_tasks:
        raise ValueError(f"Task must be one of {valid_tasks}")

    valid_avgs = ["macro", "micro", "weighted"]
    if avg not in valid_avgs:
        raise ValueError(f"Average must be one of {valid_avgs}")

    # กรณี y_true ผ่านการทำ One-Hot Encoder มาแล้ว
    if y_true.ndim > 1 and y_true.size(1) > 1:
        y_true = torch.argmax(y_true, dim=1)

    acc = accuracy(y_pred, y_true, task=task, num_classes=num_classes, average=avg)
    f1 = f1_score(y_pred, y_true, task=task, num_classes=num_classes, average=avg)
    prec = precision(y_pred, y_true, task=task, num_classes=num_classes, average=avg)
    rec = recall(y_pred, y_true, task=task, num_classes=num_classes, average=avg)

    return float(acc), float(f1), float(prec), float(rec)

In [ ]:
def training(model,
             dataloader,
             criterion,
             optimizer,
             device,
             num_classes,
             epoch,
             total_epochs,
             task="multiclass",
             avg="macro"
             ):
    model.train()

    kbar = pkbar.Kbar(target=len(dataloader),
                      epoch=epoch,
                      num_epochs=total_epochs,
                      width=30, always_stateful=True)

    running_loss = 0.0
    total_samples = 0

    all_preds = []
    all_labels = []

    for i, (inputs, labels) in enumerate(dataloader):
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        if task == "multiclass":
            labels = labels.long()
            if labels.ndim == 2:
                labels = labels.squeeze(1)
        else:
            labels = labels.float()
            if task == "binary" and labels.ndim == 1:
                labels = labels.unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size

        if task == "multiclass":
            # CrossEntropyLoss
            batch_preds = torch.argmax(outputs, dim=1)
        elif task == "binary":
            # BCEWithLogitsLoss
            probs = torch.sigmoid(outputs.view(-1))
            batch_preds = (probs > 0.5).float()
            labels = labels.view(-1)
        else:
            raise ValueError(f"Task {task} not supported")

        batch_preds = batch_preds.detach().cpu()
        batch_labels = labels.detach().cpu()

        all_preds.append(batch_preds)
        all_labels.append(batch_labels)

        # Watch update real time
        current_lr = optimizer.param_groups[0]['lr']
        b_acc, b_f1, b_prec, b_rec = performance_metrics(batch_preds, batch_labels, num_classes, task, avg)

        kbar.update(i, values=[
            ('loss', loss.item()),
            ('accuracy', b_acc),
            ('f1', b_f1),
            ('precision', b_prec),
            ('recall', b_rec),
            ('lr', current_lr)
        ])

    num_classes = None if task == "binary" else num_classes
    if total_samples == 0:
        return {key: 0.0 for key in ['loss', 'accuracy', 'f1', 'precision', 'recall', 'lr']}

    epoch_preds = torch.cat(all_preds)
    epoch_labels = torch.cat(all_labels)

    train_loss = running_loss / total_samples
    train_acc, train_f1, train_prec, train_rec = performance_metrics(epoch_preds, epoch_labels, num_classes, task, avg)

    current_lr = optimizer.param_groups[0]['lr']

    kbar.add(1, values=[
        ('loss', train_loss),
        ('accuracy', train_acc),
        ('f1', train_f1),
        ('precision', train_prec),
        ('recall', train_rec),
        ('lr', current_lr)
    ])

    return {
        'loss': train_loss,
        'accuracy': train_acc,
        'f1': train_f1,
        'precision': train_prec,
        'recall': train_rec,
        'lr': current_lr
    }

In [ ]:
def evaluate(model,
               dataloader,
               criterion,
               device,
               num_classes,
               task="multiclass",
               avg="macro",
               verbose=True
               ):
    model.eval()

    kbar = None
    if verbose:
        total_batches = len(dataloader)
        kbar = pkbar.Kbar(target=total_batches, width=30, always_stateful=True)

    running_loss = 0.0
    total_samples = 0
    all_preds = []
    all_labels = []

    with torch.inference_mode():
        for batch_idx, (inputs, labels) in enumerate(dataloader):
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            if task == "multiclass":
                labels = labels.long()
                if labels.ndim == 2:
                    labels = labels.squeeze(1)
            else:
                labels = labels.float()
                if task == "binary" and labels.ndim == 1:
                    labels = labels.unsqueeze(1)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            batch_size = inputs.size(0)
            running_loss += loss.item() * batch_size
            total_samples += batch_size

            if task == "multiclass":
                # CrossEntropyLoss
                batch_preds = torch.argmax(outputs, dim=1)
            elif task == "binary":
                # BCEWithLogitsLoss
                probs = torch.sigmoid(outputs.view(-1))
                batch_preds = (probs > 0.5).float()
                labels = labels.view(-1)
            else:
                raise ValueError(f"Task {task} not supported")

            batch_preds = batch_preds.detach().cpu()
            batch_labels = labels.detach().cpu()

            all_preds.append(batch_preds)
            all_labels.append(batch_labels)

            b_acc, b_f1, b_prec, b_rec = performance_metrics(batch_preds, batch_labels, num_classes, task, avg)

            if kbar is not None:
                kbar.update(batch_idx, values=[
                    ('loss', loss.item()),
                    ('accuracy', b_acc),
                    ('f1', b_f1),
                    ('precision', b_prec),
                    ('recall', b_rec)
                ])

    num_classes = None if task == "binary" else num_classes
    if total_samples == 0:
        return {key: 0.0 for key in ['loss', 'accuracy', 'f1', 'precision', 'recall']}

    epoch_preds = torch.cat(all_preds)
    epoch_labels = torch.cat(all_labels)

    eval_loss = running_loss / total_samples

    eval_acc, eval_f1, eval_prec, eval_rec = performance_metrics(
        epoch_preds, epoch_labels, num_classes, task, avg
    )

    if kbar is not None:
        kbar.add(1, values=[
            ('loss', eval_loss),
            ('accuracy', eval_acc),
            ('f1', eval_f1),
            ('precision', eval_prec),
            ('recall', eval_rec)
        ])

    return {
        'loss': eval_loss,
        'accuracy': eval_acc,
        'f1': eval_f1,
        'precision': eval_prec,
        'recall': eval_rec
    }

In [ ]:
def fit(model,
        train_loader,
        val_loader,
        epochs,
        criterion,
        optimizer,
        scheduler,
        device,
        num_classes,
        task="multiclass",
        avg="macro",
        initial_epoch=0,
        resume_history=None,
        callbacks=None,
        save_path=None
        ):

    if callbacks is None:
        callbacks = []
    else:
        if not isinstance(callbacks, list):
            raise TypeError("Callbacks must be a list of callbacks or a single callback.")

    if resume_history is not None:
        history = defaultdict(list, resume_history)
    else:
        history = defaultdict(list)

    start_time = time()
    actual_epoch = 0

    for epoch in range(initial_epoch, epochs):
        actual_epoch = epoch + 1
        epoch_start_time = time()

        train_metrics = training(
            model, train_loader, criterion, optimizer, device, num_classes, epoch, epochs, task, avg
        )

        val_metrics = evaluate(
            model, val_loader, criterion, device, num_classes, task, avg, verbose=False
        )

        logs = {}
        for key, value in train_metrics.items():
            logs[key] = value.item() if hasattr(value, 'item') else value

        for key, value in val_metrics.items():
            logs[f"val_{key}"] = value.item() if hasattr(value, 'item') else value

        if scheduler:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(logs['val_loss'])
            else:
                scheduler.step()

        epoch_duration = time() - epoch_start_time
        history['epoch'].append(epoch + 1)
        history['epoch_time'].append(epoch_duration) # เวลาต่อ 1 epoch

        for key, value in logs.items():
            history[key].append(value)

        print(f"val_loss: {logs['val_loss']:.4f} - val_accuracy: {logs['val_accuracy']:.4f} - val_f1: {logs['val_f1']:.4f} - val_precision: {logs['val_precision']:.4f} - val_recall: {logs['val_recall']:.4f} - time: {epoch_duration:.2f}s")

        # Callback Loop
        stop_signal = False
        for callback in callbacks:
            callback.on_epoch_end(model, epoch, logs)

            if getattr(callback, "stop_training", False):
                stop_signal = True

        if stop_signal:
            break

    total_time = time() - start_time # เวลา train + val ทุก epochs
    print("-" * 200)
    print(f"Training completed in {total_time:.2f} seconds")

    for callback in callbacks:
        if isinstance(callback, ModelCheckpoint):
            print(f"Best model found at Epoch {callback.best_epoch} ({callback.monitor}: {callback.best_score:.4f})")

    if save_path:
        save_model_state(
            save_path=save_path,
            model=model,
            epoch=actual_epoch,
            optimizer=optimizer
        )
        print(f"Model final weights has been saved to {save_path}")

    return history

In [ ]:
def predict(model, dataloader, device, task="multiclass", verbose=True):
    model.eval()
    probs_list = []

    valid_tasks = ["binary", "multiclass"]
    if task not in valid_tasks:
        raise ValueError(f"Task must be one of {valid_tasks}")

    kbar = None
    if verbose:
        total_batches = len(dataloader)
        kbar = pkbar.Kbar(target=total_batches, width=30, always_stateful=True)

    with torch.inference_mode():
        for i, batch in enumerate(dataloader):
            if isinstance(batch, (list, tuple)):
                inputs = batch[0]
            else:
                inputs = batch

            inputs = inputs.to(device, non_blocking=True)
            logits = model(inputs)

            if task == "multiclass":
                probs = torch.softmax(logits, dim=1)
            else:
                probs = torch.sigmoid(logits)
                if probs.ndim > 1 and probs.shape[-1] == 1:
                    probs = probs.view(-1)

            probs_list.append(probs.cpu())

            if kbar is not None:
                kbar.update(i + 1)

    probs = torch.cat(probs_list)
    return probs.numpy()

## **Section 4: Training the model**

In [ ]:
checkpoint_path = f"{model_dir}/{model_name}_{dataset_name}_[BEST_WEIGHT].pth"
model_path = f"{model_dir}/{model_name}_{dataset_name}_[FINAL_WEIGHT].pth"

print(f"Checkpoint path: {checkpoint_path}")
print(f"Model path: {model_path}")

In [ ]:
model_checkpoint = ModelCheckpoint(
    optimizer=optimizer,
    save_path=checkpoint_path,
    monitor="val_loss",
    mode="min",
    verbose=True
)

early_stopping = EarlyStopping(
    patience=5,
    monitor="val_loss",
    mode="min",
    verbose=True
)

In [ ]:
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=valid_loader,
    epochs=MAX_EPOCHS,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_classes=NUM_CLASSES,
    task=TASK,
    avg=AVG,
    callbacks=[model_checkpoint, early_stopping],
    save_path=model_path
)

In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv(f"{model_assets_dir}/{model_name}_{dataset_name}_history.csv", index=False)
history_df.head()

## **Section 5: Model evaluation**

### **Define helper function**

In [ ]:
def get_labels(dataloader):
    true_labels = []

    for batch in dataloader:
        if isinstance(batch, (list, tuple)):
            labels = batch[1]
        else:
            raise ValueError("Loader format not supported")

        true_labels.append(labels.cpu().numpy())

    return np.concatenate(true_labels)

In [ ]:
def plot_history(
        history,
        figsize=(12, 5),
        fontsize=16,
        save_path=None,
        title=None
        ):

    epochs = range(1, len(history["accuracy"]) + 1)
    fig = plt.figure(figsize=figsize)

    # Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["loss"], "b-", label="Training loss")
    plt.plot(epochs, history["val_loss"], "r-", label="Validation loss")
    plt.title("Training and validation loss", fontsize=fontsize)
    plt.xlabel("Epochs", fontsize=fontsize)
    plt.ylabel("Loss", fontsize=fontsize)
    plt.legend(fontsize=fontsize-2)
    plt.xticks(fontsize=fontsize-2)
    plt.yticks(fontsize=fontsize-2)

    # Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["accuracy"], "b-", label="Training accuracy")
    plt.plot(epochs, history["val_accuracy"], "r-", label="Validation accuracy")
    plt.title("Training and validation accuracy", fontsize=fontsize)
    plt.xlabel("Epochs", fontsize=fontsize)
    plt.ylabel("Accuracy", fontsize=fontsize)
    plt.legend(fontsize=fontsize-2)
    plt.xticks(fontsize=fontsize-2)
    plt.yticks(fontsize=fontsize-2)

    plt.tight_layout()

    if title:
        fig.text(0.5, 0.01, f"{title}", ha='center', fontsize=fontsize)

    if save_path:
        plt.savefig(f"{save_path}/learning_curve.png", format="png", dpi=300, bbox_inches="tight")
        plt.savefig(f"{save_path}/learning_curve.svg", format="svg", bbox_inches="tight")

    plt.show()


In [ ]:
def plot_confusion_matrix(
        cm,
        class_names=None,
        task="multiclass",
        normalize=False,
        cmap=plt.cm.Blues,
        figsize=(10, 8),
        fontsize=12,
        save_path=None,
        title=None
        ):

    valid_tasks = ["binary", "multiclass"]
    if task not in valid_tasks:
        raise ValueError(f"task must be one of {valid_tasks}")

    if cm.ndim != 2 or cm.shape[0] != cm.shape[1]:
        raise ValueError("Confusion matrix must be a 2D square matrix")

    num_classes = cm.shape[0]

    if task == "binary" and num_classes != 2:
        raise ValueError("Binary classification requires a 2x2 confusion matrix")

    if class_names is None:
        if task == "binary":
            class_names = ["Negative", "Positive"]
        else:
            class_names = [f"Class {i}" for i in range(num_classes)]

    if len(class_names) != num_classes:
        raise ValueError("Length of classes must match confusion matrix size")

    if save_path:
        os.makedirs(save_path, exist_ok=True)

    cm = cm.astype(float)

    correct = np.trace(cm)
    total = np.sum(cm)
    accuracy = correct / total if total > 0 else 0.0
    misclassification = 1.0 - accuracy

    if normalize:
        row_sum = cm.sum(axis=1, keepdims=True)
        row_sum[row_sum == 0] = 1.0
        cm = cm / row_sum

    suffix = " " + "(Normalized)" if normalize else ""
    plot_title = f"{title}\nConfusion Matrix{suffix}" if title else f"Confusion Matrix{suffix}"

    plt.figure(figsize=figsize)
    plt.imshow(cm, interpolation="nearest", cmap=cmap)
    plt.title(plot_title, fontsize=fontsize)
    plt.colorbar()

    tick_marks = np.arange(num_classes)
    plt.xticks(tick_marks, class_names, rotation=45, fontsize=fontsize)
    plt.yticks(tick_marks, class_names, fontsize=fontsize)

    fmt = ".2f" if normalize else ".0f"
    thresh = cm.max() / 2.0 if cm.size else 0.0

    for i, j in itertools.product(range(num_classes), range(num_classes)):
        plt.text(
            j, i,
            format(cm[i, j], fmt),
            ha="center",
            va="center",
            fontsize=fontsize,
            color="white" if cm[i, j] > thresh else "black"
        )

    plt.ylabel("True label", fontsize=fontsize)
    plt.xlabel(
        f"Predicted label\nAccuracy: {accuracy:.4f} | Misclass: {misclassification:.4f}",
        fontsize=fontsize
    )

    plt.tight_layout()

    if save_path:
        suffix = "normalized" if normalize else "raw"
        plt.savefig(f"{save_path}/confusion_matrix_{suffix}.png",
                    dpi=300, bbox_inches="tight")
        plt.savefig(f"{save_path}/confusion_matrix_{suffix}.svg",
                    bbox_inches="tight")

    plt.show()

In [ ]:
def plot_roc(
    y_true,
    y_pred_prob,
    class_names=None,
    task="multiclass",
    figsize=(10, 8),
    fontsize=14,
    save_path=None,
    title=None
):

    y_true = np.asarray(y_true)
    y_pred_prob = np.asarray(y_pred_prob)

    if save_path:
        os.makedirs(save_path, exist_ok=True)

    valid_tasks = ["binary", "multiclass"]
    if task not in valid_tasks:
        raise ValueError(f"task must be one of {valid_tasks}")

    plt.figure(figsize=figsize)

    if title:
        plot_title = f"{title}\nReceiver Operating Characteristic (ROC)"
    else:
        plot_title = "Receiver Operating Characteristic (ROC)"

    # Binary
    if task == "binary":

        if y_true.ndim != 1:
            raise ValueError("Binary y_true must be 1D")

        if y_pred_prob.ndim == 2:
            if y_pred_prob.shape[1] != 2:
                raise ValueError("Binary y_pred_prob must have shape (N, 2)")
            y_score = y_pred_prob[:, 1]
        else:
            y_score = y_pred_prob

        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)

        plt.plot(
            fpr, tpr,
            lw=2,
            label=f"ROC curve (AUC = {roc_auc:.4f})"
        )

        plt.plot([0, 1], [0, 1], linestyle="--", lw=1, color="gray")

    # Multiclass
    else:
        if y_pred_prob.ndim != 2:
            raise ValueError("Multiclass y_pred_prob must be 2D (N, C)")

        n_classes = y_pred_prob.shape[1]

        if class_names is None:
            class_names = [f"Class {i}" for i in range(n_classes)]

        y_true_bin = label_binarize(y_true, classes=range(n_classes))

        for i in range(n_classes):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(
                fpr, tpr,
                lw=2,
                label=f"{class_names[i]} (AUC = {roc_auc:.4f})"
            )

        fpr_micro, tpr_micro, _ = roc_curve(
            y_true_bin.ravel(), y_pred_prob.ravel()
        )
        auc_micro = auc(fpr_micro, tpr_micro)

        plt.plot(
            fpr_micro, tpr_micro,
            linestyle="--", lw=2,
            label=f"Micro-average (AUC = {auc_micro:.4f})"
        )

        auc_macro = roc_auc_score(
            y_true_bin, y_pred_prob, average="macro", multi_class="ovr"
        )

        plt.plot([0, 1], [0, 1], linestyle=":", lw=1, color="black",
                 label=f"Macro-average (AUC = {auc_macro:.4f})")

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate", fontsize=fontsize)
    plt.ylabel("True Positive Rate", fontsize=fontsize)
    plt.title(plot_title, fontsize=fontsize)
    plt.legend(loc="lower right", fontsize=fontsize)
    plt.grid(alpha=0.3)

    plt.xticks(fontsize=fontsize)
    plt.yticks(fontsize=fontsize)

    if save_path:
        plt.savefig(f"{save_path}/roc_curve.png", dpi=300, bbox_inches="tight")
        plt.savefig(f"{save_path}/roc_curve.svg", bbox_inches="tight")

    plt.show()

In [ ]:
def plot_predictions(df,
                     filepaths_col="Filepaths",
                     true_label_col="True_Label",
                     pred_label_col="Pred_Label",
                     prob_col="Prob",
                     n_images_per_class=5,
                     target_size=(224, 224),
                     figsize=(15, 3),
                     fontsize=12,
                     save_path=None,
                     title=None
                     ):

    classes = np.sort(df[true_label_col].unique())
    n_rows = len(classes)
    n_cols = n_images_per_class

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(figsize[0], figsize[1] * n_rows))

    axes = np.atleast_2d(axes)
    if axes.shape[0] != n_rows or axes.shape[1] != n_cols:
        axes = axes.reshape(n_rows, n_cols)

    if title is not None:
        fig.suptitle(title, fontsize=fontsize + 4, y=0.92)

    for row_idx, class_name in enumerate(classes):
        df_class = df[df[true_label_col] == class_name].sample(
            n=min(n_images_per_class, len(df[df[true_label_col] == class_name])),
            random_state=42
        ).reset_index(drop=True)

        for col_idx in range(n_cols):
            ax = axes[row_idx, col_idx]
            ax.axis("off")  # default off

            if col_idx < len(df_class):
                row = df_class.iloc[col_idx]
                img_path = row[filepaths_col]
                pred_class_name = row[pred_label_col]
                confidence = row[prob_col]
                color = "green" if class_name == pred_class_name else "red"

                try:
                    img = PIL.Image.open(img_path).convert('RGB')
                    img = img.resize(target_size)
                    img_display = np.array(img).astype("float32")
                    img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min() + 1e-7)
                    ax.imshow(img_display)
                except Exception as e:
                    ax.text(0.5, 0.5, f"Image Error\n{e}", ha='center', va='center')

                ax.set_title(f"True: {class_name}\nPred: {pred_class_name}\n({confidence:.2f})",
                             fontsize=fontsize, color=color)

    plt.tight_layout()
    plt.subplots_adjust(top=1.0 - (0.05 / n_rows))

    if save_path:
        plt.savefig(f"{save_path}.png", dpi=300, bbox_inches="tight")
        plt.savefig(f"{save_path}.svg", bbox_inches="tight")

    plt.show()

### **Load best weight model**

In [ ]:
checkpoints = load_model_state(
    save_path=checkpoint_path,
    model=model,
    device=device
)

### **Model performance on testset**

In [ ]:
results = evaluate(
    model=model,
    dataloader=test_loader,
    criterion=criterion,
    device=device,
    num_classes=NUM_CLASSES,
    task="multiclass",
    avg="macro",
    verbose=True
)

#### **Performance**

In [ ]:
print("Model performance on testset results:")
print("=" * 50)
for key, value in results.items():
    print(f"Test {key}: {value:.4f}")

In [ ]:
y_pred_probs = predict(
    model=model,
    dataloader=test_loader,
    device=device,
    task="multiclass",
    verbose=True
)

y_pred = np.argmax(y_pred_probs, axis=1)
y_true = get_labels(test_loader)

print(f"y_pred_probs shape: {y_pred_probs.shape}")
print(f"y_pred shape: {y_pred.shape}")
print(f"y_true shape: {y_true.shape}")

#### **Correct and Incorrect predictions**

In [ ]:
filepaths_list = [path for path, _ in test_loader.dataset.samples]

correct_records = []
incorrect_records = []

for filepath, pred_idx, true_idx, probs in zip(filepaths_list, y_pred, y_true, y_pred_probs):
    pred_idx_str = str(int(pred_idx))
    true_idx_str = str(int(true_idx))

    record = {
        "Filepaths": filepath,
        "Pred_Label": idx_to_class[pred_idx_str],
        "True_Label": idx_to_class[true_idx_str],
        "Pred_Index": int(pred_idx),
        "True_Index": int(true_idx),
        "Prob": float(probs[int(pred_idx)])
    }

    if pred_idx == true_idx:
        correct_records.append(record)
    else:
        incorrect_records.append(record)

df_correct = pd.DataFrame(correct_records)
df_correct.to_csv(f"{model_assets_dir}/{model_name}_{dataset_name}_correct_predictions.csv", index=False)
df_incorrect = pd.DataFrame(incorrect_records)
df_incorrect.to_csv(f"{model_assets_dir}/{model_name}_{dataset_name}_incorrect_predictions.csv", index=False)

In [ ]:
df_correct

In [ ]:
plot_predictions(
    df=df_correct,
    filepaths_col="Filepaths",
    true_label_col="True_Label",
    pred_label_col="Pred_Label",
    prob_col="Prob",
    n_images_per_class=6,
    target_size=IMG_SIZE,
    figsize=(18, 3),
    fontsize=12,
    save_path=f"{model_assets_dir}/correct_predictions",
)

In [ ]:
df_incorrect

In [ ]:
plot_predictions(
    df=df_incorrect,
    filepaths_col="Filepaths",
    true_label_col="True_Label",
    pred_label_col="Pred_Label",
    prob_col="Prob",
    n_images_per_class=6,
    target_size=IMG_SIZE,
    figsize=(18, 3),
    fontsize=12,
    save_path=f"{model_assets_dir}/incorrect_predictions",
)

#### **Classification report**

In [ ]:
report = classification_report(y_true, y_pred, digits=4)

report_dict = classification_report(y_true, y_pred, digits=4, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(f"{model_assets_dir}/{model_name}_{dataset_name}_classification_report.csv")

print("Classification Report")
print("=" * 100)
print(report)

#### **Learning curve**

In [ ]:
plot_history(
    history=history,
    figsize=(14, 6),
    fontsize=16,
    save_path=model_assets_dir,
    title=None
)

#### **Confusion matrix**

##### **without Normalize**

In [ ]:
cm = sklearn.metrics.confusion_matrix(y_true, y_pred)
plot_confusion_matrix(
    cm=cm,
    class_names=CLASSES_NAME,
    task=TASK,
    normalize=False,
    cmap=plt.cm.Blues,
    figsize=(10, 8),
    fontsize=14,
    save_path=model_assets_dir,
    title=None
)

##### **with Normalize**

In [ ]:
plot_confusion_matrix(
    cm=cm,
    class_names=CLASSES_NAME,
    task=TASK,
    normalize=True,
    cmap=plt.cm.Blues,
    figsize=(10, 8),
    fontsize=14,
    save_path=model_assets_dir,
    title=None
)

#### **ROC curve**

In [ ]:
plot_roc(
    y_true=y_true,
    y_pred_prob=y_pred_probs,
    class_names=CLASSES_NAME,
    task=TASK,
    figsize=(12, 6),
    fontsize=14,
    save_path=model_assets_dir,
    title=None
)

#### **Calculate Flops**

In [ ]:
flops, macs, params = calflops.calculate_flops(
    model=model,
    input_shape=(1,) + INPUT_SHAPE,
    output_precision=4
)

print("Model FLOPs:%s   MACs:%s   Params:%s \n" %(flops, macs, params))